# Phishing E-posta Tespiti ? Makine ??renmesi Demosu

Bu notebook, **phishing (oltalama) e-postalar?n?n** makine ??renmesi ile nas?l
tespit edilebilece?ini **ad?m ad?m** g?stermek i?in haz?rlanm??t?r.

### Bu notebook'ta iki veri seti kullan?yoruz

1. **Ana veri seti ? ger?ek e-posta metinleri** (`CEAS_08`): E-postan?n konusu ve
   g?vdesini i?erir. As?l modelimizi (TF-IDF + Logistic Regression) bununla kuraca??z.
2. **Kar??la?t?rma veri seti ? say?sal ?zellikler** (`ethancratchley/email-phishing-dataset`):
   E-postalar?n ham metnini de?il, ?nceden ??kar?lm?? 8 say?sal ?zelli?ini i?erir
   (kelime say?s?, link say?s?, aciliyet kelimesi say?s?...). Bunu, en sonda
   "?zellik m?hendisli?i" kar??la?t?rmas? i?in kullanaca??z.


> **Not:** ?kinci veri setinde ham metin yoktur. Bu y?zden tokenization / TF-IDF
> gibi metin i?lemleri onun ?zerinde yap?lamaz. Ders a??s?ndan bu fark?n kendisi
> g?zel bir ??renme f?rsat?d?r: "ayn? problemi iki farkl? veri temsili ile ??zmek".

---

## 0. Kurulum

In [ ]:
import warnings
warnings.filterwarnings("ignore")   

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub
from kagglehub import KaggleDatasetAdapter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, roc_auc_score,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 42
print("K?t?phaneler haz?r ?")

## 1. Phishing Nedir?

### Spam nedir?
**Spam**, istenmeyen toplu e-postalard?r. Genellikle reklam ama?l?d?r (ila?,
indirim, ?r?n tan?t?m?...). Rahats?z edicidir ama ?o?u zaman do?rudan tehlikeli
de?ildir.

### Phishing (oltalama) nedir?
**Phishing**, kullan?c?y? **kand?rarak** hassas bilgilerini (?ifre, kredi kart?,
banka bilgisi) ?almay? hedefleyen sahte e-postalard?r. Sald?rgan genellikle
g?venilir bir kurum (banka, kargo ?irketi, e-posta sa?lay?c?s?) gibi davran?r.

### ?kisi aras?ndaki fark

| | Spam | Phishing |
|---|---|---|
| **Ama?** | Reklam / tan?t?m | Bilgi ?almak, doland?rmak |
| **Tehlike** | Genelde d???k | **Y?ksek** |
| **Y?ntem** | Toplu g?nderim | Taklit + aciliyet + sahte link |




### Ger?ek hayatta neden tehlikeli?
- Sahte "banka" sayfas?na ?ifrenizi girersiniz ? hesab?n?z bo?alt?l?r.
- Kurumsal bir ?al??an kand?r?l?rsa ? **t?m ?irket** s?z?nt?ya u?rayabilir.
- Fidye yaz?l?mlar?n?n en yayg?n giri? kap?s? phishing e-postalar?d?r.

### Makine ??renmesi bunu neden ??zebilir?
Phishing e-postalar?nda **tekrar eden desenler** vard?r: aciliyet ifadeleri
("hemen do?rula", "hesab?n?z ask?ya al?nd?"), ??pheli linkler, taklit kurum
isimleri, dilbilgisi hatalar?... Bu desenler tek tek kural yazmakla yakalanmaz;
ama makine ??renmesi modeli, binlerce ?rnekten bu desenleri **kendisi ??renir**.

---

## 2. Veri Setini Y?kle (ger?ek e-posta metinleri)

Ana veri setimiz ger?ek e-postalar?n **konu (subject)** ve **g?vde (body)**
metnini i?erir. KaggleHub ?zerinden y?kl?yoruz.

In [ ]:
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "naserabdullahalam/phishing-email-dataset",
    "CEAS_08.csv",
)

print("Sat?r ve s?tun say?s?:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
df.columns

### S?tunlar?n anlam?

| S?tun | Anlam? |
|---|---|
| `sender` | G?nderen e-posta adresi |
| `receiver` | Al?c? e-posta adresi |
| `date` | G?nderim tarihi |
| `subject` | E-postan?n **konu** sat?r? |
| `body` | E-postan?n **g?vde** metni (as?l i?erik) |
| `urls` | E-postada link olup olmad??? (1/0) |
| `label` | **Hedef de?i?kenimiz**: `1 = Phishing`, `0 = G?venli (normal)` |

Bizim i?in en ?nemli s?tunlar **`subject`**, **`body`** (girdi) ve **`label`**
(??kt? / tahmin etmek istedi?imiz ?ey).

## 3. Veri Analizi (EDA ? Exploratory Data Analysis)

In [ ]:
# BO?LUK S?LME
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")

# KONU + G?VDE B?RLE??M?
df["text"] = (df["subject"] + " " + df["body"]).str.strip()

# ET?KET ENCODING
df["label_name"] = df["label"].map({0: "G?venli", 1: "Phishing"})
df[["subject", "label", "label_name"]].head()

### 3.1 S?n?f da??l?m? (phishing / normal oran?)

In [ ]:
counts = df["label_name"].value_counts()
print(counts)
print("\nPhishing oran?: %{:.1f}".format(100 * df["label"].mean()))

plt.figure()
ax = sns.countplot(data=df, x="label_name", palette=["#4C72B0", "#DD8452"])
ax.set_title("G?venli vs Phishing")
ax.set_xlabel("E-posta T?r?")
ax.set_ylabel("E-posta Say?s?")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.show()

Bu veri seti **dengeli** say?l?r (phishing ve g?venli say?lar? birbirine yak?n).
Bu iyi bir haber: dengesiz veri setlerinde model ?o?unluk s?n?f?n? ezberleyebilir.

### 3.2 Eksik veri analizi

In [ ]:
missing = df.isnull().sum()
print("S?tun baz?nda eksik de?er say?s?:\n")
print(missing)

### 3.3 E-posta uzunlu?u

In [ ]:
df["char_count"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().apply(len)

print("Ortalama karakter say?s?: {:.0f}".format(df["char_count"].mean()))
print("Ortalama kelime say?s?  : {:.0f}".format(df["word_count"].mean()))
print()
print("En UZUN e-posta : {} kelime".format(df["word_count"].max()))
print("En KISA e-posta : {} kelime".format(df["word_count"].min()))

In [ ]:
en_uzun = df.loc[df["word_count"].idxmax()]
en_kisa = df.loc[df["word_count"].idxmin()]

print("?? EN UZUN E-POSTA ({} kelime, {}) ??".format(en_uzun["word_count"], en_uzun["label_name"]))
print(en_uzun["text"][:300], "...\n")
print("?? EN KISA E-POSTA ({} kelime, {}) ??".format(en_kisa["word_count"], en_kisa["label_name"]))
print(repr(en_kisa["text"][:300]))

### 3.4 Kelime say?s? da??l?m?

E-postalar?n ?o?u k?sa, birka? tanesi ?ok uzun. Bu t?r da??l?mlarda u? de?erleri
(outlier) k?rp?p bakmak grafi?i daha okunur yapar.

In [ ]:
ust_sinir = df["word_count"].quantile(0.90)

plt.figure()
ax = sns.histplot(
    data=df[df["word_count"] <= ust_sinir],
    x="word_count", hue="label_name", bins=300, palette=["#4C72B0", "#DD8452"],
)
ax.set_title("Kelime Say?s? Da??l?m?)")
ax.set_xlabel("E-postadaki Kelime Say?s?")
ax.set_ylabel("E-posta Say?s?")
plt.show()

## 4. Ger?ek E-posta ?rnekleri

?imdi veri setinden **rastgele** 5 g?venli ve 5 phishing e-posta se?ip deneyelim.

In [ ]:
def ornekleri_goster(veri, etiket_adi, adet=5):
    ornekler = veri[veri["label_name"] == etiket_adi].sample(adet, random_state=RANDOM_SEED)
    for _, satir in ornekler.iterrows():
        konu = (satir["subject"] or "(konu yok)").strip()
        govde = satir["body"].strip().replace("\n", " ")
        print("-------------------------")
        print("Label:", etiket_adi)
        print("\nSubject:\n" + konu[:120])
        print("\nBody:\n" + govde[:400] + ("..." if len(govde) > 400 else ""))
        print()

print("===== 5 G?VENL? (NORMAL) E-POSTA =====\n")
ornekleri_goster(df, "G?venli")

In [ ]:
print("===== 5 PHISHING E-POSTA =====\n")
ornekleri_goster(df, "Phishing")

### Yorum

Phishing ?rneklerinde tipik olarak ?unlar? g?r?r?z:
- **Aciliyet hissi:** "hemen", "hesab?n?z ask?ya al?nd?", "24 saat i?inde".
- **Sahte linkler:** ger?ek kuruma benzeyen ama farkl? alan adlar?.
- **Ki?isel bilgi talebi:** ?ifre, hesap do?rulama, ?deme bilgisi.
- **Dilbilgisi/yaz?m hatalar?** ve genel "kurumsal olmayan" ton.

G?venli e-postalar ise genellikle **bilgilendirici**, belirli bir ba?lam? olan
ve ki?isel bilgi istemeyen mesajlard?r. ?rne?in:

> *"Bu, ekibin haftal?k bilgilendirme mailidir; kullan?c?dan hi?bir gizli bilgi
> istememektedir."*

vs.

> *"Bu e-posta, kullan?c?da aciliyet hissi yarat?p onu sahte bir linke t?klamaya
> y?nlendirmeye ?al???yor."*

---

## 5. Phishing Belirtileri (Anahtar Kelimeler)

Sezgimiz ?u: phishing e-postalar?nda baz? kelimeler (verify, account, password,
urgent, click, login, bank...) **daha s?k** ge?er. Bunu veriyle do?rulayal?m.

In [ ]:
# ?ncelemek istedi?imiz belirti kelimeleri
belirti_kelimeleri = [
    "http", "https", "verify", "account", "password",
    "urgent", "click", "login", "bank", "update",
]

# Her kelimenin phishing ve g?venli e-postalarda ka? kez GE?T??? e-postay? sayal?m
metin_lower = df["text"].str.lower()

satirlar = []
for kelime in belirti_kelimeleri:
    icerir = metin_lower.str.contains(kelime, regex=False)
    phishing_orani = df.loc[icerir, "label"].mean() * 100 if icerir.any() else 0
    satirlar.append({
        "kelime": kelime,
        "phishing_mailde": int(((df["label"] == 1) & icerir).sum()),
        "guvenli_mailde":  int(((df["label"] == 0) & icerir).sum()),
    })

belirti_df = pd.DataFrame(satirlar).set_index("kelime")
belirti_df

In [ ]:
# Grafikle kar??la?t?ral?m
ax = belirti_df.plot(kind="bar", figsize=(10, 5), color=["#DD8452", "#4C72B0"])
ax.set_title("Belirti Kelimelerinin Phishing vs G?venli E-postalarda G?r?lme Say?s?")
ax.set_xlabel("Anahtar Kelime")
ax.set_ylabel("Kelimeyi i?eren e-posta say?s?")
ax.legend(["Phishing e-postada", "G?venli e-postada"])
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Grafikte baz? kelimelerin (?rne?in *click*, *hhtp*, *update*) phishing
e-postalarda belirgin ?ekilde daha s?k ge?ti?ini g?rebiliriz. ??te modelimiz de
tam olarak bu t?r ipu?lar?n? ? ama **y?zlerce kelime i?in ayn? anda** ? ??renecek.

---

## 6. Veri ?n ??leme

Makine ??renmesi modeli ham metni **anlamaz**; say?lara ihtiya? duyar. Metni
say?sal bir forma ?evirmeden ?nce onu temizleriz. A?a??da her ad?m? ve **neden**
yapt???m?z? a??kl?yoruz.

| Ad?m | Ne yapar? | Neden? |
|---|---|---|
| **K???k harfe ?evirme** | `Bank` ? `bank` | "Bank" ve "bank" ayn? kelime say?ls?n |
| **Link temizleme** | `http://...` ? kald?r?l?r | Uzun URL'ler g?r?lt? yarat?r; varl??? ?nemli, i?eri?i de?il |
| **Noktalama temizleme** | `!,.?` kald?r?l?r | Kelimeye odaklanmak i?in |
| **Fazla bo?luk temizleme** | ?ok bo?luk ? tek bo?luk | D?zg?n tokenization i?in |
| **Tokenization** | metin ? kelime listesi | Modelin "birim"i kelimedir |
| **Stop words** | `the, is, and`... ??kar?l?r | Bu kelimeler ay?rt edici de?il |
| **TF-IDF** | kelime ? say?sal a??rl?k | Az ge?en ama ay?rt edici kelimeleri ?ne ??kar?r |

### TF-IDF nedir? (k?saca)
**TF-IDF = Term Frequency ? Inverse Document Frequency.**
- Bir kelime bir e-postada ?ok ge?iyorsa ? ?nemli olabilir (TF ?).
- Ama o kelime *her* e-postada ge?iyorsa (?rn. "the") ? ay?rt edici de?il (IDF ?).

Yani TF-IDF, "bu e-postaya ?zg?, ay?rt edici kelimeler" e y?ksek a??rl?k verir.

In [ ]:
def temizle(metin):
    metin = metin.lower()                           # 1) k???k harf
    metin = re.sub(r"http\S+|www\.\S+", " ", metin) # 2) linkleri kald?r
    metin = re.sub(r"[^a-z\s]", " ", metin)         # 3) harf olmayanlar? kald?r
    metin = re.sub(r"\s+", " ", metin).strip()      # 4) fazla bo?luklar? temizle
    return metin

# ?rnek: temizleme ?ncesi ve sonras?
ornek = df["text"].iloc[10]
print("?NCE :", ornek[:160])
print("\nSONRA:", temizle(ornek)[:160])

In [ ]:
df["clean_text"] = df["text"].apply(temizle)
df[["text", "clean_text"]].head(3)

> **Tokenization ve stop words'? ayr?ca kodlam?yoruz:** bunlar? birazdan
> `TfidfVectorizer` bizim i?in otomatik yapacak (`stop_words="english"` ile stop
> words'leri atacak, metni kelimelere ay?racak). K?t?phanenin haz?r ve test edilmi?
> arac?n? kullanmak, "tekerle?i yeniden icat etmekten" daha temiz bir yakla??md?r.

## 7. Model

- **%80 train**
- **%20 test:**


**Logistic Regression**, ikili s?n?fland?rma (0/1) i?in en yayg?n ve en anla??l?r algoritmad?r. "Bu e-posta phishing mi?" sorusuna **0?1 aras? bir olas?l?k** ?retir.
1. **TF-IDF** ile metni say?sal vekt?re ?evir.
2. Logistic Regression'? bu vekt?rlerle e?it.

In [ ]:
X = df["clean_text"]   # girdi: temizlenmi? metin
y = df["label"]        # ??kt?: 0/1 etiketi

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y,
)

print("E?itim k?mesi:", X_train.shape[0], "e-posta")
print("Test k?mesi  :", X_test.shape[0], "e-posta")

In [ ]:
# 1) Metni say?sala ?evir: TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english",    # ?ngilizce stop words'leri at
    max_features=5000,       # en s?k 5000 kelimeyi kullan (sadelik + h?z)
    ngram_range=(1, 2),      # tek kelimeler + ikili kelime gruplar?
)
X_train_tfidf = vectorizer.fit_transform(X_train)   # e?itimde ??REN + d?n??t?r
X_test_tfidf = vectorizer.transform(X_test)         # testte SADECE d?n??t?r

# 2) Modeli e?it
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)


print("Model e?itildi")
print("TF-IDF ?zellik (kelime) say?s?:", len(vectorizer.get_feature_names_out()))

## 9. Sonu?lar

Modeli test k?mesinde de?erlendirelim.

In [ ]:
# Test k?mesinde tahmin
y_pred = model.predict(X_test_tfidf)
y_proba = model.predict_proba(X_test_tfidf)[:, 1]   # phishing olas?l???

print("Accuracy : {:.4f}".format(accuracy_score(y_test, y_pred)))
print("Precision: {:.4f}".format(precision_score(y_test, y_pred)))
print("Recall   : {:.4f}".format(recall_score(y_test, y_pred)))
print("F1 Score : {:.4f}".format(f1_score(y_test, y_pred)))

### Metrikler ne anlama geliyor?

- **Accuracy (Do?ruluk):** T?m tahminlerin ne kadar? do?ru? Basit ama dengesiz
  verilerde yan?lt?c? olabilir.
- **Precision (Kesinlik):** "Phishing" dediklerimin ne kadar? **ger?ekten**
  phishing? D???kse ? ?ok say?da **yanl?? alarm** (g?venli maili phishing sanmak).
- **Recall (Duyarl?l?k):** Ger?ek phishinglerin ne kadar?n? **yakalad?m**?
  D???kse ? tehlikeli phishingler **g?zden ka??yor**.
- **F1 Score:** Precision ve Recall'un dengeli ortalamas?. Tek bir "genel ba?ar?"
  say?s? isteyince buna bakar?z.

> G?venlikte genellikle **Recall** ?ok ?nemlidir: bir phishing'i ka??rman?n
> bedeli, bir g?venli maile yanl??l?kla tak?lmaktan ?ok daha a??rd?r.

### Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=["G?venli", "Phishing"]))

### Confusion Matrix (Kar???kl?k Matrisi)

Bu matris, modelin **nerede** hata yapt???n? g?sterir:
- Sol ?st / sa? alt ? **do?ru** tahminler.
- Sa? ?st ? g?venliyi phishing sand?k (yanl?? alarm).
- Sol alt ? phishing'i ka??rd?k (en tehlikeli hata).

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["G?venli", "Phishing"],
            yticklabels=["G?venli", "Phishing"])
plt.title("Confusion Matrix")
plt.xlabel("Tahmin Edilen")
plt.ylabel("Ger?ek")
plt.show()

### ROC Curve

**ROC e?risi**, modelin farkl? e?ik de?erlerinde do?ru/yanl?? alarm dengesini
g?sterir. E?ri sol-?st k??eye ne kadar yak?nsa model o kadar iyidir.
**AUC** (e?ri alt?nda kalan alan) 1.0'a yakla?t?k?a m?kemmel demektir; 0.5 ise
rastgele tahmin seviyesidir.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure()
plt.plot(fpr, tpr, color="#DD8452", lw=2, label="ROC (AUC = {:.3f})".format(auc))
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Rastgele tahmin")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate (Yanl?? Alarm Oran?)")
plt.ylabel("True Positive Rate (Yakalama Oran?)")
plt.legend(loc="lower right")
plt.show()

## 10. Kendi E-postan? Dene!

?imdi modeli **canl?** kullanal?m. A?a??daki `email` de?i?kenine istedi?iniz
metni yaz?n; model phishing olup olmad???n? ve olas?l???n? s?ylesin.

In [ ]:
def phishing_mi(email_metni):
    temiz = temizle(email_metni)                 # 6. b?l?mdeki ayn? temizleme
    vec = vectorizer.transform([temiz])           # TF-IDF'e ?evir
    tahmin = model.predict(vec)[0]                # 0 / 1
    olasilik = model.predict_proba(vec)[0][1]     # phishing olas?l???

    etiket = "Phishing" if tahmin == 1 else "G?venli"
    print("Prediction\n")
    print(" ", etiket)
    print("\nProbability (phishing): %{:.0f}".format(olasilik * 100))

    # Neden b?yle dedi?ine dair basit bir a??klama: hangi belirti kelimeleri var?
    bulunan = [k for k in belirti_kelimeleri if k in temiz]
    print("\nA??klama:")
    if tahmin == 1:
        print("  Model bunu phishing olarak i?aretledi.")
        if bulunan:
            print("  Dikkat ?eken kelimeler:", ", ".join(bulunan))
        print("  Aciliyet + ki?isel bilgi/link talebi tipik phishing desenidir.")
    else:
        print("  Model bunu g?venli buldu: tehdit/aciliyet deseni zay?f.")

In [ ]:
# ??? Buraya istedi?iniz e-postay? yazabilirsiniz ???
email = """
Dear Customer,

We detected unusual activity on your account. Please verify your account
immediately to avoid suspension. Click the link below and confirm your
password and banking details within 24 hours.

http://secure-verify-login.example.com

Bank Security Team
"""

phishing_mi(email)

In [ ]:
# ?kinci ?rnek: normal, g?venli bir e-posta
email_normal = """
Hi team,

Just a reminder that our weekly project sync is tomorrow at 10:00 AM.
The agenda and meeting notes are attached. Let me know if you have anything
to add.

Thanks,
Alex
"""

phishing_mi(email_normal)

?ki ?rnekte de modelin mant?kl? kararlar verdi?ini g?r?yoruz: aciliyet + ?ifre/link
talebi i?eren mail **phishing**, s?radan bir toplant? hat?rlatmas? ise **g?venli**.

---

## 11. Bonus ? Farkl? Bir Yakla??m: Say?sal ?zellikler

?imdiye kadar e-postan?n **ham metnini** (TF-IDF ile) kulland?k. Peki ya metni
hi? g?rmeden, sadece birka? **say?sal ?zet** ile ?al??sayd?k?

Ba?ta bahsetti?imiz ikinci veri seti (`ethancratchley/email-phishing-dataset`)
tam olarak bunu yap?yor: her e-postay? **8 say?** ile temsil ediyor:
kelime say?s?, link say?s?, aciliyet kelimesi say?s?, yaz?m hatas? say?s?...

**"Feature Engineering"**

In [ ]:
df_num = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "ethancratchley/email-phishing-dataset",
    "email_phishing_data.csv",
)
print("Boyut:", df_num.shape)
print("S?tunlar:", list(df_num.columns))
df_num.head()

### 11.1 Phishing'ler hangi ?zelliklerde farkl?la??yor?

Her ?zelli?in phishing ve g?venli e-postalardaki **ortalamas?n?** kar??la?t?ral?m.

In [ ]:
ortalama = df_num.groupby("label").mean().T
ortalama.columns = ["G?venli (0)", "Phishing (1)"]
print(ortalama.round(2))

ax = ortalama.plot(kind="barh", figsize=(9, 6), color=["#4C72B0", "#DD8452"])
ax.set_title("?zellik Ortalamalar?: G?venli vs Phishing")
ax.set_xlabel("Ortalama De?er")
plt.tight_layout()
plt.show()

Bu 8 say?, phishing'i ay?rt etmekte zay?f ??nk? data de?erleri birbirine yak?n ama deneyece?iz.

### 11.2 Bu say?sal ?zelliklerle model

Bu veri seti ?ok **dengesiz** (phishing sadece ~%1.3). Adil bir kar??la?t?rma i?in
s?n?flar? dengeleyip (e?it say?da phishing ve g?venli) k???k bir model e?itelim.

In [ ]:
# Dengeli alt k?me: t?m phishing + e?it say?da g?venli
phishing_rows = df_num[df_num["label"] == 1]
safe_rows = df_num[df_num["label"] == 0].sample(len(phishing_rows), random_state=RANDOM_SEED)
balanced = pd.concat([phishing_rows, safe_rows]).sample(frac=1, random_state=RANDOM_SEED)

Xn = balanced.drop(columns="label")
yn = balanced["label"]

Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(
    Xn, yn, test_size=0.20, random_state=RANDOM_SEED, stratify=yn
)

scaler = StandardScaler().fit(Xn_tr)
num_model = LogisticRegression(max_iter=1000).fit(scaler.transform(Xn_tr), yn_tr)

yn_pred = num_model.predict(scaler.transform(Xn_te))
print("SAYISAL ?ZELL?K MODEL? (dengeli):")
print("  Accuracy : {:.3f}".format(accuracy_score(yn_te, yn_pred)))
print("  Precision: {:.3f}".format(precision_score(yn_te, yn_pred)))
print("  Recall   : {:.3f}".format(recall_score(yn_te, yn_pred)))
print("  F1 Score : {:.3f}".format(f1_score(yn_te, yn_pred)))

Le? gibi bir sonu? ortaya ??kt?, direk ham verilerden (ger?ek mailler) ile ?al??mak ve model e?itmek bizim i?in daha ba?ar?l?d?r.